# Full Pipeline — All 7 Ablation Variants — PyTorch
### MedAI Diagnose | CNN + NLP + PEPA

```
A1 — Random Forest           (sklearn baseline)
A2 — CNN only
A3 — NLP only
A4 — CNN + NLP concat fusion
A5 — CNN + NLP mean fusion
A6 — CNN + NLP + PEPA  (no dropout)
A7 — CNN + NLP + PEPA + Dropout  ← PROPOSED
```

In [1]:
# Cell 1 — GPU Setup + Imports
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = True

Device : cuda
GPU    : NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
# Cell 2 — All Sub-Module Definitions
# (same as individual notebooks, consolidated here for pipeline assembly)

# ── NLP Encoder ──────────────────────────────────────────────
class NLPEncoder(nn.Module):
    def __init__(self, input_dim=500, embed_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),       nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, embed_dim), nn.ReLU()
        )
    def forward(self, x): return self.net(x)


# ── CNN Encoder ──────────────────────────────────────────────
class CNNEncoder(nn.Module):
    def __init__(self, embed_dim=128, freeze_base=True):
        super().__init__()
        base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.avgpool  = base.avgpool
        if freeze_base:
            for p in self.features.parameters(): p.requires_grad = False
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, embed_dim), nn.ReLU()
        )
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return self.head(x)


# ── PEPA Module ──────────────────────────────────────────────
class PEPAModule(nn.Module):
    def __init__(self, img_dim=128, txt_dim=128, hidden_dim=64):
        super().__init__()
        self.gate_h  = nn.Linear(img_dim + txt_dim, hidden_dim)
        self.gate_o  = nn.Linear(hidden_dim, img_dim)
        self.txt_proj= nn.Linear(txt_dim, img_dim)
        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    def forward(self, f_img, f_txt):
        z     = torch.cat([f_img, f_txt], dim=1)
        h     = self.relu(self.gate_h(z))
        alpha = self.sigmoid(self.gate_o(h))
        w_img = alpha * f_img
        w_txt = (1 - alpha) * self.relu(self.txt_proj(f_txt))
        return w_img + w_txt

print('✓ Sub-modules defined')

✓ Sub-modules defined


In [3]:
# Cell 3 — Full Pipeline Model (A7 — Proposed)

class MedAIPipeline(nn.Module):
    """
    Full CNN + NLP + PEPA pipeline.

    Inputs:
        img  : (batch, 3, 224, 224)  — medical image or zeros
        text : (batch, tfidf_dim)    — TF-IDF symptom vector
    Output:
        logits : (batch, num_classes) — disease probabilities
    """
    def __init__(self,
                 num_classes : int,
                 tfidf_dim   : int   = 500,
                 embed_dim   : int   = 128,
                 use_dropout : bool  = True,
                 freeze_cnn  : bool  = True):
        super().__init__()

        self.cnn_enc = CNNEncoder(embed_dim=embed_dim, freeze_base=freeze_cnn)
        self.nlp_enc = NLPEncoder(input_dim=tfidf_dim, embed_dim=embed_dim)
        self.pepa    = PEPAModule(img_dim=embed_dim, txt_dim=embed_dim)

        head_layers = [
            nn.Linear(embed_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        ]
        if use_dropout:
            head_layers.append(nn.Dropout(0.4))
        head_layers += [
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        ]
        self.classifier = nn.Sequential(*head_layers)

    def forward(self, img, text):
        f_img  = self.cnn_enc(img)           # (batch, 128)
        f_txt  = self.nlp_enc(text)          # (batch, 128)
        fused  = self.pepa(f_img, f_txt)     # (batch, 128)
        logits = self.classifier(fused)      # (batch, num_classes)
        return logits


# Test build
NUM_CLS   = 50    # example
TFIDF_DIM = 500
model_A7  = MedAIPipeline(num_classes=NUM_CLS, tfidf_dim=TFIDF_DIM).to(device)

total     = sum(p.numel() for p in model_A7.parameters())
trainable = sum(p.numel() for p in model_A7.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')
print(f'Frozen params   : {total-trainable:,}  (EfficientNet base)')

Total params    : 4,905,326
Trainable params: 897,778
Frozen params   : 4,007,548  (EfficientNet base)


In [4]:
# Cell 4 — All 6 Deep Learning Variants

# ── A2: CNN only ─────────────────────────────────────────────
class ModelA2_CNNOnly(nn.Module):
    def __init__(self, num_classes, embed_dim=128):
        super().__init__()
        self.cnn = CNNEncoder(embed_dim=embed_dim)
        self.clf = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, img, text=None): return self.clf(self.cnn(img))


# ── A3: NLP only ─────────────────────────────────────────────
class ModelA3_NLPOnly(nn.Module):
    def __init__(self, num_classes, tfidf_dim=500, embed_dim=128):
        super().__init__()
        self.nlp = NLPEncoder(input_dim=tfidf_dim, embed_dim=embed_dim)
        self.clf = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, img=None, text=None): return self.clf(self.nlp(text))


# ── A4: CNN + NLP concat fusion ───────────────────────────────
class ModelA4_Concat(nn.Module):
    def __init__(self, num_classes, tfidf_dim=500, embed_dim=128):
        super().__init__()
        self.cnn = CNNEncoder(embed_dim=embed_dim)
        self.nlp = NLPEncoder(input_dim=tfidf_dim, embed_dim=embed_dim)
        self.clf = nn.Sequential(
            nn.Linear(embed_dim * 2, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, img, text):
        fused = torch.cat([self.cnn(img), self.nlp(text)], dim=1)
        return self.clf(fused)


# ── A5: CNN + NLP mean fusion ─────────────────────────────────
class ModelA5_Mean(nn.Module):
    def __init__(self, num_classes, tfidf_dim=500, embed_dim=128):
        super().__init__()
        self.cnn = CNNEncoder(embed_dim=embed_dim)
        self.nlp = NLPEncoder(input_dim=tfidf_dim, embed_dim=embed_dim)
        self.clf = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, img, text):
        fused = (self.cnn(img) + self.nlp(text)) / 2
        return self.clf(fused)


# ── A6: PEPA no dropout ───────────────────────────────────────
# Same as A7 but use_dropout=False
# model_A6 = MedAIPipeline(num_classes, tfidf_dim, use_dropout=False)


# ── Build all and show param counts ──────────────────────────
variants = {
    'A2 CNN only':        ModelA2_CNNOnly(NUM_CLS),
    'A3 NLP only':        ModelA3_NLPOnly(NUM_CLS, TFIDF_DIM),
    'A4 Concat fusion':   ModelA4_Concat(NUM_CLS, TFIDF_DIM),
    'A5 Mean fusion':     ModelA5_Mean(NUM_CLS, TFIDF_DIM),
    'A6 PEPA no dropout': MedAIPipeline(NUM_CLS, TFIDF_DIM, use_dropout=False),
    'A7 Full PEPA':       MedAIPipeline(NUM_CLS, TFIDF_DIM, use_dropout=True),
}

print(f'{"Variant":<25} {"Total":>12} {"Trainable":>12}')
print('-' * 52)
for name, m in variants.items():
    t = sum(p.numel() for p in m.parameters())
    r = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:<25} {t:>12,} {r:>12,}')

print('\n✓ All 6 deep learning variants ready')

Variant                          Total    Trainable
----------------------------------------------------
A2 CNN only                  4,441,262      433,714
A3 NLP only                    494,642      494,642
A4 Concat fusion             4,896,302      888,754
A5 Mean fusion               4,863,534      855,986
A6 PEPA no dropout           4,905,326      897,778
A7 Full PEPA                 4,905,326      897,778

✓ All 6 deep learning variants ready


In [5]:
# Cell 5 — Verify Forward Pass on GPU

model_A7 = model_A7.to(device)
model_A7.eval()

with torch.no_grad():
    dummy_img  = torch.randn(4, 3, 224, 224).to(device)
    dummy_text = torch.randn(4, TFIDF_DIM).to(device)
    output     = model_A7(dummy_img, dummy_text)

print(f'Input  img  : {dummy_img.shape}')
print(f'Input  text : {dummy_text.shape}')
print(f'Output      : {output.shape}  → (batch, {NUM_CLS} classes)')
print(f'Device      : {output.device}')

# Probabilities
probs = torch.softmax(output, dim=1)
print(f'Prob sum    : {probs[0].sum().item():.4f}  (should be 1.0)')
print('\n✓ Full pipeline forward pass successful on', device)

Input  img  : torch.Size([4, 3, 224, 224])
Input  text : torch.Size([4, 500])
Output      : torch.Size([4, 50])  → (batch, 50 classes)
Device      : cuda:0
Prob sum    : 1.0000  (should be 1.0)

✓ Full pipeline forward pass successful on cuda
